# Unsupervised Clustering Analysis

This notebook runs clustering experiments (KMeans, Gaussian Mixture, DBSCAN) on the extracted hive features, evaluates cluster quality, visualizes PCA embeddings, and saves model outputs for downstream interpretation.

In [1]:
from pathlib import Path
import sys

import pandas as pd
import matplotlib.pyplot as plt

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(PROJECT_ROOT / 'src') not in sys.path:
    sys.path.append(str(PROJECT_ROOT / 'src'))

from clustering import (
    load_feature_frame,
    run_experiments,
    save_clustering_outputs,
    summarize_cluster_distribution,
    plot_pca_scatter,
)

ASSIGNMENTS_PATH = PROJECT_ROOT / 'outputs' / 'models' / 'cluster_assignments.parquet'
METRICS_PATH = PROJECT_ROOT / 'outputs' / 'models' / 'cluster_metrics.csv'
FIG_DIR = PROJECT_ROOT / 'outputs' / 'figures'
FIG_DIR.mkdir(parents=True, exist_ok=True)

In [3]:
feature_df = load_feature_frame(
    audio_path=PROJECT_ROOT / 'outputs' / 'features' / 'audio_hourly_features.parquet',
    modulation_path=PROJECT_ROOT / 'outputs' / 'features' / 'modulation_hourly_features.parquet',
    multimodal_path=PROJECT_ROOT / 'outputs' / 'features' / 'multimodal_hourly_features.parquet',
)

print(f'Loaded feature rows: {len(feature_df)}')
print(f'Columns: {feature_df.shape[1]}')
feature_df.head()

FileNotFoundError: No feature table found. Run feature extraction first.

In [ ]:
assignments, metrics = run_experiments(
    feature_df,
    n_clusters=4,
    random_state=42,
    dbscan_eps=1.2,
    dbscan_min_samples=12,
)

save_clustering_outputs(assignments, metrics, assignments_path=ASSIGNMENTS_PATH, metrics_path=METRICS_PATH)

print(f'Saved assignments: {ASSIGNMENTS_PATH}')
print(f'Saved metrics: {METRICS_PATH}')

In [ ]:
metrics

## PCA Visualizations

In [ ]:
plot_pca_scatter(assignments, color_col='hive', title='PCA colored by hive', output_path=FIG_DIR / 'pca_by_hive.png')
plot_pca_scatter(assignments, color_col='cluster_kmeans', title='PCA colored by KMeans cluster', output_path=FIG_DIR / 'pca_by_kmeans_cluster.png')
plot_pca_scatter(assignments, color_col='cluster_gmm', title='PCA colored by GMM cluster', output_path=FIG_DIR / 'pca_by_gmm_cluster.png')
plot_pca_scatter(assignments, color_col='cluster_dbscan', title='PCA colored by DBSCAN cluster', output_path=FIG_DIR / 'pca_by_dbscan_cluster.png')

plt.show()

In [ ]:
dist_kmeans = summarize_cluster_distribution(assignments, 'cluster_kmeans')
dist_gmm = summarize_cluster_distribution(assignments, 'cluster_gmm')
dist_dbscan = summarize_cluster_distribution(assignments, 'cluster_dbscan')

print('KMeans distribution by hive')
display(dist_kmeans.head(30))

print('GMM distribution by hive')
display(dist_gmm.head(30))

print('DBSCAN distribution by hive')
display(dist_dbscan.head(30))

In [ ]:
assignments.head()